# ***`Final Results`***
---
---
***Models:***
- Mistral
- LLama
- GPT-4o

In [24]:
import pandas as pd
pd.set_option('display.max_columns', None) # display all columns

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [57]:
df = pd.read_csv('../data/original_data/newsroom_judged.csv')

columns_to_keep = [
    # Human Judgements
    'informativeness_scores', 'relevance_scores', 'fluency_scores', 'coherence_scores',
    # GPT
    'GPT_informativeness_as_a_judge', 'GPT_relevance_as_a_judge',
    'GPT_fluency_as_a_judge', 'GPT_coherence_as_a_judge',
    # LLAMA
    'LLAMA_informativeness_as_a_judge', 'LLAMA_relevance_as_a_judge',
    'LLAMA_fluency_as_a_judge', 'LLAMA_coherence_as_a_judge',
    # MISTRAL
    'MISTRAL_informativeness_as_a_judge', 'MISTRAL_relevance_as_a_judge',
    'MISTRAL_fluency_as_a_judge', 'MISTRAL_coherence_as_a_judge'
]

df = df.loc[:, columns_to_keep]
print(df.shape)
df.head()


(420, 16)


,informativeness_scores,relevance_scores,fluency_scores,coherence_scores,GPT_informativeness_as_a_judge,GPT_relevance_as_a_judge,GPT_fluency_as_a_judge,GPT_coherence_as_a_judge,LLAMA_informativeness_as_a_judge,LLAMA_relevance_as_a_judge,LLAMA_fluency_as_a_judge,LLAMA_coherence_as_a_judge,MISTRAL_informativeness_as_a_judge,MISTRAL_relevance_as_a_judge,MISTRAL_fluency_as_a_judge,MISTRAL_coherence_as_a_judge
0,"[4, 3, 1]","[4, 5, 1]","[3, 5, 3]","[4, 4, 3]",Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,1,1,1,1,1 (low),1 (low),1 (low),1
1,"[4, 5, 4]","[4, 5, 5]","[3, 5, 5]","[3, 5, 4]",Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,4,5,4,4,4.5,5,5,5
2,"[3, 5, 4]","[4, 5, 3]","[4, 5, 3]","[4, 5, 4]",Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,3,4,4,4,3.5 (The summary captures the main points of t...,4.5 (The summary is very consistent with the d...,5 (high),4.5 (The summary is concise and accurately rep...
3,"[3, 3, 3]","[3, 4, 4]","[3, 2, 4]","[3, 2, 3]",Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,1,2,2,2,2,3,2,2
4,"[3, 4, 5]","[4, 3, 3]","[4, 3, 4]","[3, 4, 4]",Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,1,1,1,1,2,3,1 (low) - The summary is incomplete and lacks ...,1


- cast human scores to list & check whethjer allways 3 judges present

In [58]:
import ast

human_scores = [col for col in df.columns if 'scores' in col]

import ast

# Safely evaluate only if it's a string
for col in human_scores:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# allways three judges?
for col in human_scores:
    lengths = df[col].apply(len)

    # See value counts of list lengths
    print(lengths.value_counts())

    # Check if all lists have the same length
    all_same_length = lengths.nunique() == 1
    print(f"All lists same length? {all_same_length}")


informativeness_scores
3    420
Name: count, dtype: int64
All lists same length? True
relevance_scores
3    420
Name: count, dtype: int64
All lists same length? True
fluency_scores
3    420
Name: count, dtype: int64
All lists same length? True
coherence_scores
3    420
Name: count, dtype: int64
All lists same length? True


- reak up human judgements into columns (human_#1,...,human_#3)

In [59]:
# break up human judgements into columns (human_#1,...,human_#3)
for col in human_scores:
    metric_name = col.split('_')[0]
    for i in range(1, 4):
        new_col_name = f'{metric_name}_human_#{i}'
        df[new_col_name] = df[col].apply(lambda x: x[i-1])

df.drop(columns=human_scores, inplace=True)

df.head(3)

,GPT_informativeness_as_a_judge,GPT_relevance_as_a_judge,GPT_fluency_as_a_judge,GPT_coherence_as_a_judge,LLAMA_informativeness_as_a_judge,LLAMA_relevance_as_a_judge,LLAMA_fluency_as_a_judge,LLAMA_coherence_as_a_judge,MISTRAL_informativeness_as_a_judge,MISTRAL_relevance_as_a_judge,MISTRAL_fluency_as_a_judge,MISTRAL_coherence_as_a_judge,informativeness_human_#1,informativeness_human_#2,informativeness_human_#3,relevance_human_#1,relevance_human_#2,relevance_human_#3,fluency_human_#1,fluency_human_#2,fluency_human_#3,coherence_human_#1,coherence_human_#2,coherence_human_#3
0,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,1,1,1,1,1 (low),1 (low),1 (low),1,4,3,1,4,5,1,3,5,3,4,4,3
1,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,4,5,4,4,4.5,5,5,5,4,5,4,4,5,5,3,5,5,3,5,4
2,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,3,4,4,4,3.5 (The summary captures the main points of t...,4.5 (The summary is very consistent with the d...,5 (high),4.5 (The summary is concise and accurately rep...,3,5,4,4,5,3,4,5,3,4,5,4


- extract numerical scores from llm outputs:

In [60]:
import re

def extract_valid_rating(series: pd.Series) -> pd.Series:
    """
    Extract leading number from strings in a Series,
    keeping only values between 0 and 6 inclusive.
    Non-matching or out-of-range values are returned as NaN.

    Parameters
    ----------
    series : pd.Series

    Returns
    -------
    pd.Series
        Series of integers in [0, 6] or NaN.
    """
    pattern = re.compile(r'^\s*(\d+)')

    values = []
    for item in series:
        text = str(item)
        match = pattern.match(text)
        if match:
            value = int(match.group(1))
            if 0 <= value <= 6:
                values.append(value)
            else:
                values.append(np.nan)
        else:
            values.append(np.nan)

    return pd.Series(values, index=series.index)




model_columns = [col for col in df.columns if 'as_a_judge' in col]
# Drop rows where any model column contains 'Evaluation Error'
mask = df[model_columns].apply(lambda row: row.astype(str).str.contains("Evaluation Error")).any(axis=1)
df = df[~mask]


for col in model_columns:
    df[col] = extract_valid_rating(df[col]).astype(int)

# drop nan
df.dropna(inplace=True)


df.head(3)

,GPT_informativeness_as_a_judge,GPT_relevance_as_a_judge,GPT_fluency_as_a_judge,GPT_coherence_as_a_judge,LLAMA_informativeness_as_a_judge,LLAMA_relevance_as_a_judge,LLAMA_fluency_as_a_judge,LLAMA_coherence_as_a_judge,MISTRAL_informativeness_as_a_judge,MISTRAL_relevance_as_a_judge,MISTRAL_fluency_as_a_judge,MISTRAL_coherence_as_a_judge,informativeness_human_#1,informativeness_human_#2,informativeness_human_#3,relevance_human_#1,relevance_human_#2,relevance_human_#3,fluency_human_#1,fluency_human_#2,fluency_human_#3,coherence_human_#1,coherence_human_#2,coherence_human_#3
7,1,1,1,1,2,2,1,2,3,1,2,1,3,1,1,4,1,1,5,3,1,5,1,1
8,1,1,1,1,2,2,2,3,4,3,3,3,3,1,4,4,2,3,4,3,3,4,1,4
9,1,1,2,1,2,2,2,2,5,5,4,4,2,1,5,2,5,4,3,5,5,2,2,5


## ***`Diversity of Opinion`***
---
### ***1. Human-Human Disagreement***


In [61]:
import numpy as np
from itertools import combinations

def compute_human_human_disagreements(df, human_cols):
    """
    Compute all pairwise absolute differences between human raters per case.

    Parameters:
        df (pd.DataFrame): DataFrame containing human ratings
        human_cols (list of str): List of column names for human ratings

    Returns:
        np.ndarray: Flattened array of all pairwise absolute disagreements
    """
    absolute_differences = []

    if len(human_cols) < 2:
        raise ValueError("At least two human columns are required")

    for _, row in df[human_cols].iterrows():
        ratings = row.dropna().values
        for a, b in combinations(ratings, 2):
            absolute_differences.append(abs(a - b))

    return np.array(absolute_differences)


human_informativeness_cols = [col for col in df.columns if 'informativeness_human' in col]
human_relevance_cols = [col for col in df.columns if 'relevance_human' in col]
human_fluency_cols = [col for col in df.columns if 'fluency_human' in col]
human_coherence_cols = [col for col in df.columns if 'coherence_human' in col]

human_disagreements = {
    'informativeness': compute_human_human_disagreements(df, human_informativeness_cols),
    'relevance': compute_human_human_disagreements(df, human_relevance_cols),
    'fluency': compute_human_human_disagreements(df, human_fluency_cols),
    'coherence': compute_human_human_disagreements(df, human_coherence_cols)
}

# print mean and std for each array in human_disagreements
for metric, disagreements in human_disagreements.items():
    print(f"{metric}: mean={disagreements.mean():.2f}, std={disagreements.std():.2f}")

informativeness: mean=1.05, std=0.95
relevance: mean=1.15, std=1.05
fluency: mean=1.45, std=1.11
coherence: mean=1.32, std=1.07


### ***2. Human-LLM Disagreement***

In [64]:
def compute_llm_human_disagreements(df: pd.DataFrame, human_cols: list[str], llm_col: str) -> np.ndarray:
    """
    Compute absolute differences between LLM judge and each human judge per case.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing ratings.
    human_cols : list of str
        Column names for human judges.
    llm_col : str
        Column name for the LLM judge's ratings.

    Returns
    -------
    np.ndarray
        Array of absolute differences (LLM vs. each human) across all cases.
    """
    absolute_differences = []

    if len(human_cols) < 2:
        raise ValueError("At least two human columns are required")

    for _, row in df.iterrows():
        llm_rating = row[llm_col]

        # Skip cases where LLM rating is missing
        if pd.isna(llm_rating):
            continue

        for col in human_cols:
            human_rating = row[col]
            if not pd.isna(human_rating):
                absolute_differences.append(abs(human_rating - llm_rating))

    return np.array(absolute_differences)


llm_human_disagreements = {

}
for model in ['MISTRAL', 'LLAMA', 'GPT']:
    for metric in ['informativeness', 'relevance', 'fluency', 'coherence']:
        llm_human_disagreements[f'{model}_{metric}'] = compute_llm_human_disagreements(
            df, human_informativeness_cols, f'{model}_{metric}_as_a_judge'
        )

# print mean and std for each array in llm_human_disagreements
for metric, disagreements in llm_human_disagreements.items():
    print(f"{metric}: mean={disagreements.mean():.2f}, std={disagreements.std():.2f}")

MISTRAL_informativeness: mean=1.02, std=0.99
MISTRAL_relevance: mean=1.00, std=0.95
MISTRAL_fluency: mean=1.20, std=1.02
MISTRAL_coherence: mean=1.16, std=1.03
LLAMA_informativeness: mean=1.57, std=1.05
LLAMA_relevance: mean=1.33, std=1.05
LLAMA_fluency: mean=1.36, std=1.10
LLAMA_coherence: mean=1.19, std=0.96
GPT_informativeness: mean=1.73, std=1.10
GPT_relevance: mean=1.39, std=1.10
GPT_fluency: mean=1.36, std=1.13
GPT_coherence: mean=1.60, std=1.13
